# Case 1 — River Level Monitoring: Sensor Drift

## Assignment context

This notebook is part of the Smart Monitoring System assignment for Master students in Civil Engineering and Territorial Protection.

**Monitoring objective:** Flood monitoring and early warning  
**Main issue:** Progressive sensor drift

Tasks:

1. inspect the raw sensor data;
2. detect the issue visually;
3. detect the issue statistically or with ML;
4. decide whether to correct, flag or preserve the observations;
5. prepare the dataset for ingestion, analysis, dashboarding and alerting in istSOS4Things.


## Case description

A river level sensor is installed near a critical bridge section. The monitoring system supports flood early warning and operational decision-making.

The dataset contains a progressive drift: the sensor slowly overestimates the river level. This is dangerous because it may produce false alarms or distort flood forecasting models.


## 1. Import libraries and load the dataset


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = "dataset_1_river_level_drift.csv"
VALUE_COL = "river_level_m"

df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)
df.head()


## 2. First inspection


In [ ]:
print(df.info())
print("\nMissing values:")
print(df.isna().sum())
print("\nSummary statistics:")
display(df.describe())


## 3. Visual inspection of raw data


In [ ]:
plt.figure(figsize=(12,4))
plt.plot(df["timestamp"], df[VALUE_COL], marker=".", linewidth=1)
plt.title(f"Raw time series: {VALUE_COL}")
plt.xlabel("Time")
plt.ylabel(VALUE_COL)
plt.grid(True)
plt.show()


## Detection and correction strategy

Suggested checks:

- plot the raw time series;
- compute a rolling mean;
- estimate a linear trend;
- compare the raw signal with a detrended version.

Possible treatment:

- subtract the estimated drift component;
- keep both raw and corrected values;
- add a quality flag.


## 4. Rolling mean and trend inspection


In [ ]:
df["rolling_24h"] = df[VALUE_COL].rolling(window=24, center=True).mean()

plt.figure(figsize=(12,4))
plt.plot(df["timestamp"], df[VALUE_COL], label="raw", alpha=0.6)
plt.plot(df["timestamp"], df["rolling_24h"], label="24h rolling mean", linewidth=2)
plt.title("River level with rolling mean")
plt.xlabel("Time")
plt.ylabel("River level [m]")
plt.legend()
plt.grid(True)
plt.show()


## 5. Estimate and remove linear drift


In [ ]:
x = np.arange(len(df))
valid = df[VALUE_COL].notna()
coef = np.polyfit(x[valid], df.loc[valid, VALUE_COL], 1)
trend = np.polyval(coef, x)

df["estimated_trend"] = trend
df["river_level_corrected_m"] = df[VALUE_COL] - (trend - trend[0])

print("Estimated trend slope per time step:", coef[0])

plt.figure(figsize=(12,4))
plt.plot(df["timestamp"], df[VALUE_COL], label="raw")
plt.plot(df["timestamp"], df["river_level_corrected_m"], label="corrected")
plt.title("Raw vs drift-corrected river level")
plt.xlabel("Time")
plt.ylabel("River level [m]")
plt.legend()
plt.grid(True)
plt.show()


## 6. Create quality flag and export cleaned dataset


In [ ]:
df["quality_flag"] = "raw"
df.loc[df.index > len(df) * 0.5, "quality_flag"] = "suspected_drift"

cleaned = df[["timestamp", "river_level_m", "river_level_corrected_m", "quality_flag"]]
cleaned.to_csv("cleaned_dataset_1_river_level_drift.csv", index=False)
cleaned.head()


## Final questions

1. What is the main data quality issue or hazardous event?
2. Which visual method was most useful?
3. Which statistical or ML method was most useful?
4. Which observations should be corrected, removed, flagged or preserved?
5. What would be a suitable alerting rule for an operational dashboard?
6. How would you model this dataset in SensorThings API?
   - Thing
   - Location
   - Sensor
   - ObservedProperty
   - Datastream
   - Observation
